Импорт библиотек

In [12]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Библиотеки подключены")

Библиотеки подключены


Загрузка данных и разделение

In [13]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("обучающая:", X_train.shape)
print("тестовая:", X_test.shape)

обучающая: (120, 4)
тестовая: (30, 4)


Создание и обучение моделей

In [14]:
models = {
    "логистическая регрессия": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "k ближайших соседей": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    ),
    "дерево решений": DecisionTreeClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "модель обучена")

логистическая регрессия модель обучена
k ближайших соседей модель обучена
дерево решений модель обучена


Оценка качества

In [15]:
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"Алгоритм": name, "Accuracy": acc})
    print("\n", name)
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

results_df = pd.DataFrame(results)
print(results_df.sort_values("Accuracy", ascending=False))


 логистическая регрессия
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30


 k ближайших соседей
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30


 дерево решений
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virgini

Матрица ошибок

In [16]:
y_pred_tree = models["дерево решений"].predict(X_test)
print(confusion_matrix(y_test, y_pred_tree))

[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


kNN с n_neighbors = 3 и 7

In [17]:
for k in (3, 5, 7):
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"k={k}: accuracy={acc:.4f}")

k=3: accuracy=0.9333
k=5: accuracy=0.9333
k=7: accuracy=0.9667


Дерево с max_depth=2

In [18]:
tree_limited = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_limited.fit(X_train, y_train)
print("max_depth=2:", accuracy_score(y_test, tree_limited.predict(X_test)))

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)
print("без ограничения:", accuracy_score(y_test, tree_full.predict(X_test)))

max_depth=2: 0.9333333333333333
без ограничения: 0.9333333333333333


Сводная таблица Accuracy

In [19]:
summary = []
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    summary.append({"Алгоритм": name, "Accuracy": round(acc, 4)})

print(pd.DataFrame(summary))

                  Алгоритм  Accuracy
0  логистическая регрессия    0.9333
1      k ближайших соседей    0.9333
2           дерево решений    0.9333


Сравнение с MLP из ПР №7

In [20]:
from sklearn.neural_network import MLPClassifier

# те же X_train/X_test (без масштабирования — делаем pipeline)
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42)
)
mlp.fit(X_train, y_train)
print("MLP (10 нейронов):", accuracy_score(y_test, mlp.predict(X_test)))

MLP (10 нейронов): 0.9666666666666667
